# Шаг 20. Проверка выбросов и аномалий

Цель на данном этапе — не удалить всё, что выбивается из среднего, а выявить и описать природу этих значений. В предметной области исторических миниатюр многие "выбросы" являются не ошибками, а важными особенностями.

Будет осуществлена роверка границ логики: рейтинг не может быть > 50, количество фигур не может быть < 0. 
Выявленные выбросы необходимо исправить. 
Крайние значения необходмо объяснить. 

In [2]:
import pandas as pd

# Загружаем данные уровня набора
df = pd.read_csv('df_sets.csv', dtype={
    'release_year': 'Int64',
    'aggregate_rating': 'Int64',
    'num_figures': 'Int64',
    'decade': 'Int64',
    'years_since_release': 'Int64'
})

print("=== 1. Выбросы в годе выпуска (release_year) ===")
# Слишком старые (до эры массового пластика ~1950) или из будущего (> 2026)
outliers_year = df[(df['release_year'] < 1950) | (df['release_year'] > 2026)]
print(f"Найдено наборов: {len(outliers_year)}")
if len(outliers_year) > 0:
    print(outliers_year[['header', 'release_year']].head())
    print("🔍 ПРИРОДА: Это редкое, но реальное событие или артефакты парсинга (неизвестная дата). Это особый сегмент 'ретро', удалять нельзя.")

print("\n=== 2. Выбросы в рейтинге качества (aggregate_rating) ===")
# Рейтинг является суммой 5 оценок по 10 баллов, диапазон строго [5, 50]
outliers_rating = df[(df['aggregate_rating'] < 5) | (df['aggregate_rating'] > 50)]
print(f"Найдено наборов: {len(outliers_rating)}")
if len(outliers_rating) > 0:
    print(outliers_rating[['header', 'aggregate_rating']].head())
    print("🔍 ПРИРОДА: Вероятная ошибка ввода или парсинга. Такие значения не имеют физического смысла и требуют ручной проверки или замены на NaN.")

print("\n=== 3. Выбросы в количестве фигур (num_figures) ===")
# Наборы с аномально большим количеством фигур (например, > 100)
outliers_figures = df[df['num_figures'] > 100]
print(f"Найдено наборов: {len(outliers_figures)}")
if len(outliers_figures) > 0:
    print(outliers_figures[['header', 'num_figures']].head())
    print("🔍 ПРИРОДА: Это не ошибка, а особый сегмент. Это крупные сэмплеры или масштабные диорамы. Необходимо оставить без изменений.")

print("\n=== 4. Выбросы в длительности исторического периода (period_duration) ===")
# Периоды длиннее 1000 лет (условные "Древний мир" без конкретики)
outliers_duration = df[df['period_duration'] > 1000]
print(f"Найдено наборов: {len(outliers_duration)}")
if len(outliers_duration) > 0:
    print(outliers_duration[['header', 'years_from', 'years_to', 'period_duration']].head())
    print("🔍 ПРИРОДА: Реальное событие, особенность предметной области. Производители иногда выпускают наборы с широким охватом. Удалять нельзя.")

print("\n=== 5. Выбросы в давности релиза (years_since_release) ===")
# Наборы, выпущенные очень давно (более 60 лет назад)
outliers_recency = df[df['years_since_release'] > 60]
print(f"Найдено наборов: {len(outliers_recency)}")
if len(outliers_recency) > 0:
    print(outliers_recency[['header', 'release_year', 'years_since_release']].head())
    print("🔍 ПРИРОДА: Редкое, но реальное событие. Это коллекционные винтажные наборы. Они критически важны для RFM-анализа, так как показывают темы, которые производитель не переиздавал десятилетиями.")

print("\n" + "="*60)
print("ИТОГ ШАГА 20:")
print("Большинство выявленных 'выбросов' являются легитимными особенностями")
print("рынка исторических миниатюр (винтаж, сэмплеры, широкие периоды).")
print("Автоматическое удаление строк (dropna/drop) НЕ применялось, чтобы")
print("сохранить полноту каталога и честность аналитики.")
print("="*60)

=== 1. Выбросы в годе выпуска (release_year) ===
Найдено наборов: 0

=== 2. Выбросы в рейтинге качества (aggregate_rating) ===
Найдено наборов: 0

=== 3. Выбросы в количестве фигур (num_figures) ===
Найдено наборов: 9
                                                 header  num_figures
815                     Strelets Medieval Britain (071)          124
1035  Strelets Russian General Staff and Hospital (051)          115
1461                       Strelets Thin Red Line (903)          147
1527           Strelets Court and Army of Peter I (905)          108
1668      Strelets The Last Assault on Sevastopol (906)          190
🔍 ПРИРОДА: Это не ошибка, а особый сегмент. Это крупные сэмплеры или масштабные диорамы. Необходимо оставить без изменений.

=== 4. Выбросы в длительности исторического периода (period_duration) ===
Найдено наборов: 19
                                 header  years_from  years_to  period_duration
172       HYTTY Historic Figures (1003)         -70      1960         

## Результат шага
Выбросы выявлены, их природа классифицирована (реальное событие, особый сегмент, особенность предметной области). Решено не удалять их автоматически, чтобы сохранить целостность и информативность данных.